# NB03 · Semana 7 — DuckDB y Delta sobre el lake (exploración)
**Ingeniería de Datos — LCD · UCA Rosario · 2026** · grupo06 · Clima de Sídney (Open-Meteo)

Leemos bronze (Delta) y lo exploramos con SQL en DuckDB. Lo que encontramos acá es lo que limpia la capa silver (E2).

In [ ]:
import os, sys
from pathlib import Path
# Raíz del proyecto = carpeta padre de notebooks/ (en el servidor: ~/work/dags)
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
os.chdir(RAIZ)
import entorno
entorno.info()
import duckdb, pandas as pd
from deltalake import DeltaTable
dt = DeltaTable(entorno.BRONZE, storage_options=entorno.STORAGE_OPTIONS)
bronze = dt.to_pandas()
print(bronze.shape, "| versión Delta", dt.version())

## 1. Consultas analíticas con DuckDB (sobre el DataFrame, sin cargarlo a ninguna base)

In [ ]:
duckdb.sql("""
select fuente, count(*) dias, min(fecha) desde, max(fecha) hasta,
       round(avg(temperature_2m_mean),1) temp_media, round(sum(precipitation_sum)/count(*)*365,0) lluvia_anual_mm
from bronze group by fuente order by desde""").df()

## 2. Problemas reales encontrados en los datos crudos

In [ ]:
duckdb.sql("""
select fuente,
  round(avg(rain_sum),2) prom_rain_sum, round(avg(precipitation_sum),2) prom_precipitation_sum,
  count(*) filter (where precipitation_sum > 0 and precipitation_hours = 0) precip_sin_horas,
  count(*) filter (where wind_direction_10m_dominant = 360) dir_360,
  count(*) filter (where temperature_2m_min > temperature_2m_max) min_mayor_max
from bronze group by fuente""").df()

- **`rain_sum` no significa lo mismo en las dos fuentes**: en ERA5 es igual a la precipitación; en Historical Forecast excluye los chaparrones (es menor). Silver usa `precipitation_sum`.
- **Días con precipitación > 0 y 0 horas de precipitación** en ERA5: lloviznas por debajo del umbral horario. Se documenta y se usa el criterio "día lluvioso = ≥ 1 mm".
- **Dirección 360° y 0°** = mismo norte → se normaliza.

In [ ]:
duckdb.sql("""
select weather_code, count(*) filter (where fuente='archive_era5') era5, count(*) filter (where fuente='historical_forecast') hist_forecast
from bronze group by 1 order by 1""").df()

- **`weather_code` usa vocabularios distintos**: ERA5 nunca codifica niebla (45), chaparrones (80-82) ni tormentas (95-99). Por eso silver los agrupa en categorías y el tablero advierte que 'días de tormenta' antes de 2026 es 0 por la fuente, no por el clima.

## 3. Time travel sobre Delta
Delta guarda el historial: podemos leer la tabla como estaba en cualquier versión anterior.

In [ ]:
hist = pd.DataFrame(dt.history())[["version", "timestamp", "operation"]]
print(hist.head(10))
v0 = DeltaTable(entorno.BRONZE, version=0, storage_options=entorno.STORAGE_OPTIONS).to_pandas()
print("versión 0:", len(v0), "filas | hoy:", len(bronze), "filas")